In [1]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import pandas as pd

In [2]:
model_name = 'allenai/longformer-base-4096' # More detailed model, uses 4096 tokens instead of 512 like the mini
# model_name = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()
df = pd.read_csv('cleaned_dataset.csv')

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return (token_embeddings * input_mask_expanded).sum(1) / input_mask_expanded.sum(1).clamp(min=1e-9)

def get_embeddings(texts, batch_size=32):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors='pt')
        encoded = {k: v.to(device) for k, v in encoded.items()}  # move inputs to GPU
        with torch.no_grad():
            output = model(**encoded)
        embeddings = mean_pooling(output, encoded['attention_mask'])
        embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
        all_embeddings.append(embeddings.cpu().numpy())  # move back to CPU for numpy
        if i % 500 == 0:
            print(f"Processed {i}/{len(texts)}")
    return np.vstack(all_embeddings)

Loading weights:   0%|          | 0/271 [00:00<?, ?it/s]

LongformerModel LOAD REPORT from: allenai/longformer-base-4096
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 5070 Ti


In [4]:
texts = df['page_text'].tolist()
embeddings = get_embeddings(texts)
df['embeddings'] = list(embeddings)

Processed 0/10832
Processed 4000/10832
Processed 8000/10832


In [5]:
df.to_csv('cleaned_dataset_embeddings.csv')

In [7]:
df.to_pickle('df_with_embeddings.pkl')